In [1]:
import pandas as pd

Fill the universityType, programType, region, city values of the no idOSYM programs from matching universityName, faculty, departmentName rows.

In [2]:
df = pd.read_csv("fulldata_clean.csv")

columns_to_fill = ["universityType", "programType", "universityRegion"]

university_type_and_region_dict = {}
program_type_dict = {}

for _,row in df.iterrows():
    if str(row["universityName"]) != "":
        key_2 = str(row["universityName"])
        university_type_and_region_dict[key_2] = {'universityType': row['universityType'], 'universityRegion': row['universityRegion']} 
    
    if str(row["universityName"]) != "" or str(row["faculty"]) != "" or str(row["departmentName"]) != "":
        key_1 = (str(row["universityName"]), str(row["faculty"]), str(row["departmentName"]))
        program_type_dict[key_1] = {"programType": row["programType"]}
    


def fix_types_and_regions(row):
    for column in columns_to_fill:
        if pd.isnull(row[column]):
            if column == "universityType" or column == "universityRegion":
                key = str(row["universityName"])
                row[column] = university_type_and_region_dict[key][column]
            else:
                key = (str(row["universityName"]), str(row["faculty"]), str(row["departmentName"]))
                row[column] = program_type_dict[key][column]
    
    return row

df = df.apply(fix_types_and_regions, axis=1)
df.to_csv("fulldata_clean_fillMissingTypesAndRegions.csv", index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_19652\2477468898.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_clean.csv")


Filling prof assoc doc counts by retrieving from same universityName, faculty, departmentName same year if possible. If same year not possible, from previous years, if thats not possible as well from closest year. Of the same department

In [5]:
df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions.csv")

columns_of_academic_staff = ["profCount", "assoCount", "docCount"]

years = ["2019.0", "2020.0" , "2021.0", "2022.0", "2023.0", "2024.0"]

academic_staff_dict = {}

academic_staff_dict_with_id = {}

for _,row in df.iterrows():
    if (row["profCount"] == 0 and row["assoCount"] == 0 and row["docCount"] == 0) or (pd.isnull(row["profCount"]) and pd.isnull(row["assoCount"]) and pd.isnull(row["docCount"])):
        pass
    else:
        key = (str(row["academicYear"]) ,str(row["universityName"]), str(row["faculty"]), str(row["departmentName"]))
        key2 = (str(row["academicYear"]),str(row["idOSYM"]))
        if key not in academic_staff_dict:
            academic_staff_dict[key] = {"profCount":row["profCount"], "assoCount": row["assoCount"], "docCount":row["docCount"]}
        if key2 not in academic_staff_dict_with_id:
            academic_staff_dict_with_id[key2] = {"profCount":row["profCount"], "assoCount": row["assoCount"], "docCount":row["docCount"]}
        
def fill_academic_counts(row):
    if (row["profCount"] == 0 and row["assoCount"] == 0 and row["docCount"] == 0) or (pd.isnull(row["profCount"]) and pd.isnull(row["assoCount"]) and pd.isnull(row["docCount"])):
        if (not pd.isnull(row["academicYear"])):
            year = str(row["academicYear"])
            index_of_year = years.index(year)
            i = index_of_year
            found = False
            if i != -1:
                while i >= 0 and not found:
                    key = (str(years[i]),str(row["idOSYM"]))
                    if key in academic_staff_dict_with_id:
                        row["profCount"] = academic_staff_dict_with_id[key]["profCount"]
                        row["assoCount"] = academic_staff_dict_with_id[key]["assoCount"]
                        row["docCount"] = academic_staff_dict_with_id[key]["docCount"]
                        found = True
                    i -= 1
                i = index_of_year + 1
                while i< len(years) and not found:
                    key = (str(years[i]),str(row["idOSYM"]))
                    if key in academic_staff_dict_with_id:
                        row["profCount"] = academic_staff_dict_with_id[key]["profCount"]
                        row["assoCount"] = academic_staff_dict_with_id[key]["assoCount"]
                        row["docCount"] = academic_staff_dict_with_id[key]["docCount"]
                        found = True
                    i += 1
                
                
                i = index_of_year
            
                
                while i >= 0 and not found:
                    key = (str(years[i]) ,str(row["universityName"]), str(row["faculty"]), str(row["departmentName"]))
                    if key in academic_staff_dict:
                        row["profCount"] = academic_staff_dict[key]["profCount"]
                        row["assoCount"] = academic_staff_dict[key]["assoCount"]
                        row["docCount"] = academic_staff_dict[key]["docCount"]
                        found = True
                    i -= 1
                i = index_of_year + 1
                while i< len(years) and not found:
                    key = (str(years[i]) ,str(row["universityName"]), str(row["faculty"]), str(row["departmentName"]))
                    if key in academic_staff_dict:
                        row["profCount"] = academic_staff_dict[key]["profCount"]
                        row["assoCount"] = academic_staff_dict[key]["assoCount"]
                        row["docCount"] = academic_staff_dict[key]["docCount"]
                        found = True
                    i += 1
            if not found:
                row["profCount"] = "0"
                row["assoCount"] = "0"
                row["docCount"] = "0"
                    
                
        
        
    return row


df = df.apply(fill_academic_counts, axis=1)

df.to_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts.csv", index=False)
        

C:\Users\AliCe\AppData\Local\Temp\ipykernel_19652\4155143548.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions.csv")


Set the rankings of the departments which didnt get any students to that years worst ranking plus x, their admitted related, outofcity, same region,  columns to 0. 

In [6]:
df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts.csv")

columns_to_zero = ["outOfCityStudentRate", "sameRegionStudentRate", "top1AdmittedRatio", "top3AdmittedRatio", "top10AdmittedRatio","admittedGovPref", "admittedPrivPref","admittedTotalPref"]

columns_to_worst = ["baseRanking", "topRanking", "avgAdmissionRanking(TYT)", "baseAdmissionRanking(TYT)","avgAdmittedStudentPrefOrder", "baseScore", "topScore"]


worst_base_ranking = {}
worst_top_ranking = {}
worst_avg_tyt = {}
worst_base_tyt = {}
worst_avg_pref = {}
worst_base_score = {}
worst_top_score = {}

for _,row in df.iterrows():
    key = str(row["academicYear"])
    if str(row["occupiedSlots"]) != "0":
        
        if not pd.isnull(row["baseRanking"]):
            if key not in worst_base_ranking:
                worst_base_ranking[key] = int(row["baseRanking"])
            if key in worst_base_ranking and int(row["baseRanking"]) > worst_base_ranking[key]:
                worst_base_ranking[key] = int(row["baseRanking"])
        
        if not pd.isnull(row["topRanking"]):
            if key not in worst_top_ranking:
                worst_top_ranking[key] = int(row["topRanking"])
            if key in worst_top_ranking and int(row["topRanking"]) > worst_top_ranking[key]:
                worst_top_ranking[key] = int(row["topRanking"])
        
        if not pd.isnull(row["avgAdmissionRanking(TYT)"]):    
            if key not in worst_avg_tyt:
                worst_avg_tyt[key] = int(row["avgAdmissionRanking(TYT)"])
            if key in worst_avg_tyt and int(row["avgAdmissionRanking(TYT)"]) > worst_avg_tyt[key]:
                worst_avg_tyt[key] = int(row["avgAdmissionRanking(TYT)"])
          
        if not pd.isnull(row["baseAdmissionRanking(TYT)"]):  
            if key not in worst_base_tyt:
                worst_base_tyt[key] = int(row["baseAdmissionRanking(TYT)"])
            if key in worst_base_tyt and int(row["baseAdmissionRanking(TYT)"]) > worst_base_tyt[key]:
                worst_base_tyt[key] = int(row["baseAdmissionRanking(TYT)"])   
        
        if not pd.isnull(row["avgAdmittedStudentPrefOrder"]):
            if key not in worst_avg_pref:
                worst_avg_pref[key] = float(row["avgAdmittedStudentPrefOrder"])
            if key in worst_avg_pref and float(row["avgAdmittedStudentPrefOrder"]) > worst_avg_pref[key]:
                worst_avg_pref[key] = float(row["avgAdmittedStudentPrefOrder"])
        
        if not pd.isnull(row["baseScore"]) and row["baseScore"] != "0" and row["baseScore"] != "0.0":    
            if key not in worst_base_score:
                worst_base_score[key] = float(row["baseScore"])
            if key in worst_base_score and float(row["baseScore"]) < worst_base_score[key]:
                worst_base_score[key] = float(row["baseScore"])
        
        if not pd.isnull(row["topScore"]) and row["topScore"] != "0" and row["topScore"] != "0.0":   
            if key not in worst_top_score:
                worst_top_score[key] = float(row["topScore"])
            if key in worst_top_score and float(row["topScore"]) < worst_top_score[key]:
                worst_top_score[key] = float(row["topScore"])    
          


def noStudents_to_zero(row):
    if str(row["occupiedSlots"]) == "0" or  str(row["occupiedSlots"]) == "0.0":     
        for column in columns_to_zero:
            row[column] = "0"
    return row

def noStudents_to_worst(row):
    if str(row["occupiedSlots"]) == "0" or  str(row["occupiedSlots"]) == "0.0": 
        academicYear = str(row["academicYear"])
        row["baseRanking"] = worst_base_ranking[academicYear]
        row["topRanking"] = worst_top_ranking[academicYear]
        row["avgAdmissionRanking(TYT)"] = worst_avg_tyt[academicYear]
        row["baseAdmissionRanking(TYT)"] = worst_base_tyt[academicYear]
        row["avgAdmittedStudentPrefOrder"] = worst_avg_pref[academicYear]
        row["baseScore"] = worst_base_score[academicYear]
        row["topScore"] = worst_top_score[academicYear]
    return row


df = df.apply(noStudents_to_zero, axis=1)
df = df.apply(noStudents_to_worst, axis=1)


df.to_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled.csv", index=False)



C:\Users\AliCe\AppData\Local\Temp\ipykernel_19652\2731128328.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts.csv")


Find the empty studentCounts similar to the academic personal count.

In [9]:
df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled.csv")

columns_of_academic_staff = ["currentStudentCount"]

years = ["2019.0", "2020.0" , "2021.0", "2022.0", "2023.0", "2024.0"]

current_student_dict = {}

current_student_dict_with_id = {}

for _,row in df.iterrows():
    if (row["currentStudentCount"] == 0) or (pd.isnull(row["currentStudentCount"])):
        pass
    else:
        key = (str(row["academicYear"]) ,str(row["universityName"]), str(row["faculty"]), str(row["departmentName"]))
        key2 = (str(row["academicYear"]),str(row["idOSYM"]))
        if key not in current_student_dict:
            current_student_dict[key] = {"currentStudentCount":row["currentcurrentStudentCount"]}
        if key2 not in current_student_dict_with_id:
            current_student_dict_with_id[key2] = {"currentStudentCount":row["currentStudentCount"]}
        
def fill_current_student_counts(row):
    if (row["currentStudentCount"] == 0) or (pd.isnull(row["currentStudentCount"])):
        if (not pd.isnull(row["academicYear"])):
            year = str(row["academicYear"])
            index_of_year = years.index(year)
            i = index_of_year
            found = False
            if i != -1:
                while i >= 0 and not found:
                    key = (str(years[i]),str(row["idOSYM"]))
                    if key in current_student_dict_with_id:
                        row["currentStudentCount"] = current_student_dict_with_id[key]["currentStudentCount"]
                        found = True
                    i -= 1
                i = index_of_year + 1
                while i< len(years) and not found:
                    key = (str(years[i]),str(row["idOSYM"]))
                    if key in current_student_dict_with_id:
                        row["currentStudentCount"] = current_student_dict_with_id[key]["currentStudentCount"]
                        found = True
                    i += 1
                
                
                i = index_of_year
            
                
                while i >= 0 and not found:
                    key = (str(years[i]) ,str(row["universityName"]), str(row["faculty"]), str(row["departmentName"]))
                    if key in current_student_dict:
                        row["currentStudentCount"] = current_student_dict[key]["currentStudentCount"]
                        found = True
                    i -= 1
                i = index_of_year + 1
                while i< len(years) and not found:
                    key = (str(years[i]) ,str(row["universityName"]), str(row["faculty"]), str(row["departmentName"]))
                    if key in current_student_dict:
                        row["currentStudentCount"] = current_student_dict[key]["currentStudentCount"]
                        found = True
                    i += 1
                    
                
        
        
    return row


df = df.apply(fill_current_student_counts, axis=1)

df.to_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled.csv", index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_19652\365658113.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled.csv")


For empty or 0 baseScore topScore where 1 or more people entered it will be calculated from the average of same programs baseScore and topScore

In [15]:
df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled.csv")

columns_to_fix = ["baseRanking", "topRanking", "avgAdmissionRanking(TYT)", "baseAdmissionRanking(TYT)", "baseScore", "topScore"]

base_ranking_average_dict = {}
top_ranking_average_dict = {}
average_tyt_average_dict = {}
base_tyt_average_dict = {}
base_score_average_dict = {}
top_score_average_dict = {}


base_ranking_average = 0.0
top_ranking_average = 0.0
average_tyt_average = 0.0
base_tyt_average = 0.0
base_score_average = 0.0
top_score_average = 0.0
count = 0

for _,row in df.iterrows():
    count += 1
    key = (str(row["universityName"]),str(row["faculty"]),str(row["departmentName"]),str(row["scholarshipRate"]),str(row["language"]))
    
    if not pd.isnull(row["baseRanking"]):
        if key not in base_ranking_average_dict:
            base_ranking_average_dict[key] = {"baseRanking":float(row["baseRanking"]), "count": 1.0}
        if key in base_ranking_average_dict:
            count = base_ranking_average_dict[key]["count"]
            base_ranking_average_dict[key]["baseRanking"] =  (base_ranking_average_dict[key]["baseRanking"] + float(row["baseRanking"])) / (count + 1)
            base_ranking_average_dict[key]["count"] = base_ranking_average_dict[key]["count"] + 1
        base_ranking_average = (base_ranking_average + float(row["baseRanking"])) / count
            
            
    if not pd.isnull(row["topRanking"]) and (str(row["topRanking"]) != "0" or str(row["topRanking"]) != "0.0"):
        if key not in top_ranking_average_dict:
            top_ranking_average_dict[key] = {"topRanking":float(row["topRanking"]), "count": 1.0}
        if key in top_ranking_average_dict:
            count = top_ranking_average_dict[key]["count"]
            top_ranking_average_dict[key]["topRanking"] =  (top_ranking_average_dict[key]["topRanking"] + float(row["topRanking"])) / (count + 1)
            top_ranking_average_dict[key]["count"] = top_ranking_average_dict[key]["count"] + 1
        top_ranking_average = (top_ranking_average + float(row["topRanking"])) / count
    
    if not pd.isnull(row["avgAdmissionRanking(TYT)"]) and (str(row["avgAdmissionRanking(TYT)"]) != "0" or str(row["avgAdmissionRanking(TYT)"]) != "0.0"):
        if key not in average_tyt_average_dict:
            average_tyt_average_dict[key] = {"avgAdmissionRanking(TYT)":float(row["avgAdmissionRanking(TYT)"]), "count": 1.0}
        if key in average_tyt_average_dict:
            count = average_tyt_average_dict[key]["count"]
            average_tyt_average_dict[key]["avgAdmissionRanking(TYT)"] =  (average_tyt_average_dict[key]["avgAdmissionRanking(TYT)"] + float(row["avgAdmissionRanking(TYT)"])) / (count + 1)
            average_tyt_average_dict[key]["count"] = average_tyt_average_dict[key]["count"] + 1
        average_tyt_average = (average_tyt_average + float(row["avgAdmissionRanking(TYT)"])) / count
    
    if not pd.isnull(row["baseAdmissionRanking(TYT)"]) and ( str(row["baseAdmissionRanking(TYT)"]) != "0" or str(row["baseAdmissionRanking(TYT)"]) != "0.0"):
        if key not in base_tyt_average_dict:
            base_tyt_average_dict[key] = {"baseAdmissionRanking(TYT)":float(row["baseAdmissionRanking(TYT)"]), "count": 1.0}
        if key in base_tyt_average_dict:
            count = base_tyt_average_dict[key]["count"]
            base_tyt_average_dict[key]["baseAdmissionRanking(TYT)"] =  (base_tyt_average_dict[key]["baseAdmissionRanking(TYT)"] + float(row["baseAdmissionRanking(TYT)"])) / (count + 1)
            base_tyt_average_dict[key]["count"] = base_tyt_average_dict[key]["count"] + 1
        base_tyt_average = (base_tyt_average + float(row["baseAdmissionRanking(TYT)"])) / count
            
            
    if not pd.isnull(row["baseScore"]) and (str(row["baseScore"]) != "0" or str(row["baseScore"]) != "0.0"):
        if key not in base_score_average_dict:
            base_score_average_dict[key] = {"baseScore":float(row["baseScore"]), "count": 1.0}
        if key in base_score_average_dict:
            count = base_score_average_dict[key]["count"]
            base_score_average_dict[key]["baseScore"] =  (base_score_average_dict[key]["baseScore"] + float(row["baseScore"])) / (count + 1)
            base_score_average_dict[key]["count"] = base_score_average_dict[key]["count"] + 1
        base_score_average = (base_score_average + float(row["baseScore"])) / count

    if not pd.isnull(row["topScore"]) and (str(row["topScore"]) != "0" or str(row["topScore"]) != "0.0"):
        if key not in top_score_average_dict:
            top_score_average_dict[key] = {"topScore":float(row["topScore"]), "count": 1.0}
        if key in top_score_average_dict:
            count = top_score_average_dict[key]["count"]
            top_score_average_dict[key]["topScore"] =  (top_score_average_dict[key]["topScore"] + float(row["topScore"])) / (count + 1)
            top_score_average_dict[key]["count"] = top_score_average_dict[key]["count"] + 1
        top_score_average = (top_score_average + float(row["topScore"])) / count




def missing_rank_score_average(row):
    key = (str(row["universityName"]),str(row["faculty"]),str(row["departmentName"]),str(row["scholarshipRate"]),str(row["language"]))
    if str(row["occupiedSlots"]) != "0" or str(row["occupiedSlots"]) != "0.0":
        for column in columns_to_fix:
            fixed = False
            if pd.isnull(row[column]) or str(row[column]) == "0" or str(row[column]) == "0.0":
                if column == "baseRanking":
                    if key in base_ranking_average_dict:
                        row[column] = base_ranking_average_dict[key][column]
                        fixed = True
                    if not fixed:
                        row[column] = base_ranking_average
                if column == "topRanking":
                    if key in top_ranking_average_dict:
                        row[column] = top_ranking_average_dict[key][column]
                        fixed = True
                    if not fixed:
                        row[column] = top_ranking_average
                if column == "avgAdmissionRanking(TYT)":
                    if key in average_tyt_average_dict:
                        row[column] = average_tyt_average_dict[key][column]
                        fixed = True
                    if not fixed:
                        row[column] = average_tyt_average
                if column == "baseAdmissionRanking(TYT)":
                    if key in base_tyt_average_dict:
                        row[column] = base_tyt_average_dict[key][column]
                        fixed = True
                    if not fixed:
                        row[column] = base_tyt_average
                if column == "baseScore":
                    if key in base_score_average_dict:
                        row[column] = base_score_average_dict[key][column]
                        fixed = True
                    if not fixed:
                        row[column] = base_score_average
                if column == "topScore":
                    if key in top_score_average_dict:
                        row[column] = top_score_average_dict[key][column]
                        fixed = True
                    if not fixed:
                        row[column] = top_score_average
            
    return row

    
df = df.apply(missing_rank_score_average, axis=1)

df.to_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled_missingRankScoreFilled.csv", index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_19652\1496456588.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled.csv")


Dropping sameRegionStudentRate,top3PreferenceRatio,top9PreferenceRatio,admittedGovPref,admittedPrivPref columns and dropping rows without idOSYM.

In [16]:
import pandas as pd

# Load the DataFrame
df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled_missingRankScoreFilled.csv")

# Columns to drop
columns_to_drop = ["sameRegionStudentRate", "top3PreferenceRatio", "top9PreferenceRatio", "admittedGovPref", "admittedPrivPref"]

# Drop the columns
df = df.drop(columns=columns_to_drop)
df = df.dropna(subset=['idOSYM'])


df.to_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled_missingRankScoreFilled_droppedRowsColumns.csv", index=False)

C:\Users\AliCe\AppData\Local\Temp\ipykernel_19652\2710248311.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled_missingRankScoreFilled.csv")


Filling in final missing studentCounts from faculties average studentCount if that doesnt work, universities average student count

In [ ]:
df = pd.read_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled_missingRankScoreFilled_droppedRowsColumns.csv")

average_student_count = {}
average_student_count_uni = {}

for _,row in df.iterrows():
    key = (str(row["universityName"]), str(row["faculty"]))
    if key not in average_student_count:
        average_student_count[key] = {"currentStudentCount":float(row["currentStudentCount"]), "count": 1.0 }
    else:
        count = average_student_count[key]["count"] + 1
        average_student_count[key]["currentStudentCount"] = (average_student_count[key]["currentStudentCount"] + float(row["currentStudentCount"])) / count
        average_student_count[key]["count"] = average_student_count[key]["count"] + 1
        
    key2 = str(row["universityName"])
    if key2 not in average_student_count_uni:
        average_student_count_uni[key] = {"currentStudentCount":float(row["currentStudentCount"]), "count": 1.0 }
    else:
        count = average_student_count_uni[key]["count"] + 1
        average_student_count_uni[key]["currentStudentCount"] = (average_student_count_uni[key]["currentStudentCount"] + float(row["currentStudentCount"])) / count
        average_student_count_uni[key]["count"] = average_student_count_uni[key]["count"] + 1
        
        
def fill_missing_student_count(row):
    
    if pd.isnull(row["currentStudentCount"]) or str(row["currentStudentCount"]) == "0" or str(row["currentStudentCount"]) == "0.0":
        fixed = False
        key1 = (str(row["universityName"]), str(row["faculty"]))
        if key1 in average_student_count:
            row["currentStudentCount"] = average_student_count[key]["currentStudentCount"]
            fixed = True
        key2 = str(row["universityName"])
        if not fixed and key2 in average_student_count_uni:
            row["currentStudentCount"] = average_student_count_uni[key]["currentStudentCount"]
            fixed = True
        if not fixed:
            row["currentStudentCount"] = "0"
    return row
        
df = df.apply(fill_missing_student_count, axis=1)

df.to_csv("fulldata_clean_fillMissingTypesAndRegions_filledAcademicCounts_0admittedFilled_studentCountFilled_missingRankScoreFilled_droppedRowsColumns_fullstudentCount.csv")